In [26]:
# pip install chromadb sentence-transformers pandas numpy


In [14]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer


In [6]:
documents = {
"Travel": """
you need to contact admin for refund.
Airlines allow passengers to check in online 24 hours before departure.
Baggage allowance depends on ticket class and route.
Flight cancellations may be eligible for refunds or rescheduling.
International travel requires valid passports and visas.
Delays caused by weather are not always compensated.
""",


"HR": """

Employees are entitled to annual leave as per company policy.
Sick leave requires medical certification beyond two consecutive days.
Performance appraisals are conducted annually.
Remote work is allowed subject to manager approval.
Code of conduct violations may lead to disciplinary action.
""",


"Hospital": """
Patients must register at the reception before consultation.
Emergency services are available 24 hours a day.
Medical records are confidential and protected by law.
Insurance coverage varies by provider and treatment.
Visiting hours are restricted in intensive care units.
""",


"E-commerce": """
Customers can return products within 7 days of delivery.
Refunds are processed after quality inspection.
Cash on delivery is available in select locations.
Discount coupons have expiration dates.
Order tracking information is sent via email and SMS.
""",


"Finance": """
Banks offer savings and current accounts.
Loan eligibility depends on credit score and income.
Interest rates vary by loan type and tenure.
Online banking transactions require two-factor authentication.
Investment products are subject to market risks.
"""
}

In [15]:
def chunk_text(text):
    return [line.strip() for line in text.strip().split("\n") if line.strip()]


In [16]:
rows = []

for domain, text in documents.items():
    for chunk in chunk_text(text):
        rows.append({"domain": domain, "text": chunk})

df = pd.DataFrame(rows)
df


,domain,text
0,Travel,you need to contact admin for refund.
1,Travel,Airlines allow passengers to check in online 2...
2,Travel,Baggage allowance depends on ticket class and ...
3,Travel,Flight cancellations may be eligible for refun...
4,Travel,International travel requires valid passports ...
5,Travel,Delays caused by weather are not always compen...
6,HR,Employees are entitled to annual leave as per ...
7,HR,Sick leave requires medical certification beyo...
8,HR,Performance appraisals are conducted annually.
9,HR,Remote work is allowed subject to manager appr...


In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")


In [19]:
embeddings = model.encode(df["text"].tolist()).tolist()


In [20]:
client = chromadb.Client()
vector_db = client.create_collection("rag")


In [21]:
vector_db.add(
    documents=df["text"].tolist(),
    embeddings=embeddings,
    metadatas=df[["domain"]].to_dict("records"),
    ids=[str(i) for i in range(len(df))]
)


In [22]:
class Retriever:
    def __init__(self, model, db):
        self.model = model
        self.db = db

    def search(self, query, top_k=3, threshold=0.4):
        query_embedding = self.model.encode(query).tolist()
        results = self.db.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        texts = results["documents"][0]
        scores = results["distances"][0]
        domains = results["metadatas"][0]

        output = []
        for t, s, d in zip(texts, scores, domains):
            similarity = 1 - s
            if similarity >= threshold:
                output.append({
                    "domain": d["domain"],
                    "text": t,
                    "similarity": similarity
                })
        return pd.DataFrame(output)


In [23]:
retriever = Retriever(model, vector_db)


In [24]:
retriever.search("What is the refund policy for flights?")


,domain,text,similarity
0,Travel,Flight cancellations may be eligible for refun...,0.409729


In [25]:
retriever.search("What is the rule for sick leave?")


,domain,text,similarity
0,HR,Sick leave requires medical certification beyo...,0.444694
